In [12]:
import awkward as ak
import uproot
import numpy as np

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

import sys
sys.path.append("..")
from src.processors.Processor import Processor
from src.processors.GenMatch import GenMatch

# Dataset

In [13]:
import os

filepath = '/home/pku/fudawei/production/samples/HGamma'
gen_masses = [dir for dir in os.listdir(filepath) if dir.endswith('GeV')] + ['total']
samples = {gen_mass: [] for gen_mass in gen_masses}
for (current_path, dirs, files) in os.walk(top='/home/pku/fudawei/production/samples/HGamma', topdown=True):
    ## topdown=False means depth-first
    postfix = current_path.split('/')[-1]
    if postfix in gen_masses:
        samples[postfix] = os.path.join(current_path, 'total.root')
    
samples

{'1000GeV': '/home/pku/fudawei/production/samples/HGamma/1000GeV/total.root',
 '1600GeV': '/home/pku/fudawei/production/samples/HGamma/1600GeV/total.root',
 '2200GeV': '/home/pku/fudawei/production/samples/HGamma/2200GeV/total.root',
 '2600GeV': '/home/pku/fudawei/production/samples/HGamma/2600GeV/total.root',
 '600GeV': '/home/pku/fudawei/production/samples/HGamma/600GeV/total.root',
 '1200GeV': '/home/pku/fudawei/production/samples/HGamma/1200GeV/total.root',
 '800GeV': '/home/pku/fudawei/production/samples/HGamma/800GeV/total.root',
 '900GeV': '/home/pku/fudawei/production/samples/HGamma/900GeV/total.root',
 '3500GeV': '/home/pku/fudawei/production/samples/HGamma/3500GeV/total.root',
 '1400GeV': '/home/pku/fudawei/production/samples/HGamma/1400GeV/total.root',
 '3000GeV': '/home/pku/fudawei/production/samples/HGamma/3000GeV/total.root',
 '700GeV': '/home/pku/fudawei/production/samples/HGamma/700GeV/total.root',
 '1800GeV': '/home/pku/fudawei/production/samples/HGamma/1800GeV/total.r

# From ROOT files to events

In [14]:
from concurrent.futures import process


events = {
    s: NanoEventsFactory.from_root(file=samples[s], schemaclass=NanoAODSchema).events() for s in samples
}
output = {e: None for e in events}

### Signal sample size

In [15]:
for gen_mass in events:
    print(gen_mass, 'events num:', len(events[gen_mass]))

1000GeV events num: 50000
1600GeV events num: 50000
2200GeV events num: 50000
2600GeV events num: 50000
600GeV events num: 50000
1200GeV events num: 50000
800GeV events num: 50000
900GeV events num: 50000
3500GeV events num: 50000
1400GeV events num: 50000
3000GeV events num: 50000
700GeV events num: 50000
1800GeV events num: 50000
2000GeV events num: 50000
2400GeV events num: 50000
total events num: 750000


# Running

In [16]:
for (key, e) in events.items():
    p = Processor()
    output[key]=p.process(events=e, deltaR_cut=0)
    ak.to_parquet(array=output[key], where=f'../output/{key}_output.parq')

/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
